# TrustSkin AI — Phase 5: Controlled Ablation Ladder, Multi-Seed Robustness & Deep Ensemble

**CS320 Fall 2026 — Project P08: Reliable Recognition of Difficult Skin Lesions**  
**Repository:** https://github.com/SikandarHussain6858/TrustSkinAI

This notebook runs the controlled Phase 5 experiments on a Google Colab GPU (~1.5–2.5 hours total on T4):
1. **Ablation R1:** Pretrained ResNet-18, plain CE, NO augmentation (`configs/abl_r1_resnet_ce_noaug.yaml`)
2. **Ablation R2:** R1 + dermoscopy augmentation (`configs/abl_r2_resnet_ce_aug.yaml`)
3. **Ablation R3s:** R2 + square-root dampened class weights (`configs/abl_r3s_resnet_sqrtw_aug.yaml`)
4. **3-Seed Robustness + Deep Ensemble:** Trains seeds `43` and `44` of the best R3/R3s config (chosen on **validation** performance!) and compares **Deterministic Softmax vs. MC Dropout vs. Deep Ensemble**.
5. **Statistical & Difficult-Case Suite:** Re-runs `05_compare_experiments.py` (ablation ladder) and `07_difficult_case_eval.py` (difficult subsets, bootstrap 95% CIs, paired McNemar tests).

In [ ]:
# Cell 1: Verify Colab GPU runtime
import torch
!nvidia-smi
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())

In [ ]:
# Cell 2: Clone/pull TrustSkinAI, set PROJECT_ROOT & PYTHONPATH=src
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/SikandarHussain6858/TrustSkinAI.git'
cwd = Path.cwd().resolve()
if (cwd / 'pyproject.toml').exists() and (cwd / 'configs' / 'base.yaml').exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'pyproject.toml').exists() and (cwd.parent / 'configs' / 'base.yaml').exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = Path('/content/TrustSkinAI')
    if not PROJECT_ROOT.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], check=True)
    else:
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull'], check=True)

os.chdir(PROJECT_ROOT)
src_dir = str((PROJECT_ROOT / 'src').resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
os.environ['PYTHONPATH'] = src_dir
print('PROJECT_ROOT:', Path.cwd())

In [ ]:
# Cell 3: Fast multi-stream download of MedIMeta derm.zip from Zenodo if not already present
import json
import shutil
import subprocess
import urllib.request
import zipfile
from pathlib import Path

data_root = Path('data')
derm_dir = data_root / 'derm'
zip_path = data_root / 'derm.zip'

if (derm_dir / 'info.yaml').exists():
    print('Dataset already ready at:', derm_dir.resolve())
else:
    if derm_dir.exists():
        shutil.rmtree(derm_dir)
    data_root.mkdir(parents=True, exist_ok=True)
    if shutil.which('aria2c') is None:
        !apt-get update -qq && apt-get install -y -qq aria2
    with urllib.request.urlopen('https://zenodo.org/api/records/7884735') as resp:
        record = json.loads(resp.read().decode('utf-8'))
    zip_file = next(f for f in record['files'] if f['key'].lower() == 'derm.zip')
    download_url = zip_file['links']['self']
    subprocess.run([
        'aria2c', '-x', '16', '-s', '16', '-k', '1M', '--continue=true',
        '--summary-interval=10', '-d', str(data_root), '-o', 'derm.zip', download_url
    ], check=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        top_names = {name.split('/')[0] for name in zf.namelist() if name}
    target = data_root if 'derm' in top_names else derm_dir
    target.mkdir(parents=True, exist_ok=True)
    subprocess.run(['unzip', '-q', '-o', str(zip_path), '-d', str(target)], check=True)
    zip_path.unlink(missing_ok=True)
    print('Dataset extracted cleanly!')

In [ ]:
# Cell 4: Verify single-factor config diffs (R1 -> R2 -> R3s) & run pytest
from trustskin.utils import load_config

r1 = load_config('configs/abl_r1_resnet_ce_noaug.yaml')
r2 = load_config('configs/abl_r2_resnet_ce_aug.yaml')
r3s = load_config('configs/abl_r3s_resnet_sqrtw_aug.yaml')
print('R1 -> R2 diff: dataset.augment =', r1['dataset']['augment'], '->', r2['dataset']['augment'])
print('R2 -> R3s diff: training.loss =', r2['training']['loss'], '->', r3s['training']['loss'], '(power =', r3s['training'].get('class_weight_power'), ')')

!pytest tests/ -v

In [ ]:
# Cell 5: Ablation R1 — Pretrained ResNet-18, plain CE, NO augmentation
!python scripts/03_train_improved.py --config configs/abl_r1_resnet_ce_noaug.yaml
!python scripts/04_calibrate_and_select.py \
    --config configs/abl_r1_resnet_ce_noaug.yaml \
    --checkpoint checkpoints/abl_r1_resnet_ce_noaug_best.pt

In [ ]:
# Cell 6: Ablation R2 — R1 + Dermoscopy Data Augmentation (plain CE)
!python scripts/03_train_improved.py --config configs/abl_r2_resnet_ce_aug.yaml
!python scripts/04_calibrate_and_select.py \
    --config configs/abl_r2_resnet_ce_aug.yaml \
    --checkpoint checkpoints/abl_r2_resnet_ce_aug_best.pt

In [ ]:
# Cell 7: Ablation R3s — R2 + Square-Root Dampened Class Weights (power = 0.5)
!python scripts/03_train_improved.py --config configs/abl_r3s_resnet_sqrtw_aug.yaml
!python scripts/04_calibrate_and_select.py \
    --config configs/abl_r3s_resnet_sqrtw_aug.yaml \
    --checkpoint checkpoints/abl_r3s_resnet_sqrtw_aug_best.pt

In [ ]:
# Cell 8: Multi-Seed Robustness (Seeds 43 & 44) for R3s -> Forms 3-Model Deep Ensemble
# (If validation selects linear R3 instead of R3s, switch BEST_CFG to configs/resnet18_mc.yaml)
BEST_CFG = 'configs/abl_r3s_resnet_sqrtw_aug.yaml'
SEED0_CKPT = 'checkpoints/abl_r3s_resnet_sqrtw_aug_best.pt'

!python scripts/03_train_improved.py --config {BEST_CFG} --seed 43 --exp-name r3s_seed43
!python scripts/04_calibrate_and_select.py --config {BEST_CFG} --checkpoint checkpoints/r3s_seed43_best.pt --exp-name r3s_seed43

!python scripts/03_train_improved.py --config {BEST_CFG} --seed 44 --exp-name r3s_seed44
!python scripts/04_calibrate_and_select.py --config {BEST_CFG} --checkpoint checkpoints/r3s_seed44_best.pt --exp-name r3s_seed44

In [ ]:
# Cell 9: Uncertainty Method Comparison (Deterministic vs. MC Dropout vs. 3-Seed Deep Ensemble)
!python scripts/08_ensemble_eval.py \
    --config {BEST_CFG} \
    --checkpoints {SEED0_CKPT} checkpoints/r3s_seed43_best.pt checkpoints/r3s_seed44_best.pt \
    --mc-samples 25

In [ ]:
# Cell 10: Compile Ablation Ladder, Difficult-Case Subsets, Bootstrap 95% CIs, McNemar Tests & Package Zip
!python scripts/05_compare_experiments.py --output-dir outputs
!python scripts/07_difficult_case_eval.py --output-dir outputs

!zip -rq outputs/trustskin_phase5_results.zip checkpoints outputs/tables outputs/figures outputs/logs
from pathlib import Path
import pandas as pd
bundle = Path('outputs/trustskin_phase5_results.zip')
print('Packaged Phase 5 bundle:', bundle.resolve(), f'({bundle.stat().st_size / 1e6:.2f} MB)')
for tbl in ['ablation_ladder.csv', 'uncertainty_method_comparison.csv', 'difficult_case_metrics.csv', 'mcnemar_tests.csv']:
    p = Path('outputs/tables') / tbl
    if p.exists():
        print('\n===', tbl, '===')
        display(pd.read_csv(p))